# Homicidios en Quito frente a los demás cantones

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `Tasa_Homicidios_2020-2026_5.xlsx` (y, para la verificación final, `261014_homicidios_quito_vs_cantones_v3.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "statsmodels", "openpyxl", "matplotlib"], check=False)

DATA_FILES = ["Tasa_Homicidios_2020-2026_5.xlsx"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

Quito frente al resto de cantones — homicidios intencionales 2014-2026
Pregunta: ¿qué tan seguro es Quito frente a los demás cantones y cómo evolucionó cada uno frente a Quito?
Unidad: cantón × año. Tasa = homicidios intencionales (HI) por 100.000 habitantes.
Grupo de referencia FIJO: cantones con >= 50.000 hab. en 2025 (para que el grupo no cambie entre años).
2026 es un acumulado parcial (la base no declara fecha de corte).

In [ ]:
# Carga y conciliación
import pandas as pd, numpy as np, json
raw = pd.read_excel(DATA_PATH, dtype={"DPA_CANTON *": str, "DPA_PROVIN": str})
raw.columns = ["cod","canton","prov","provincia","anio","pob","hi","tasa","blanca","fuego","otras","masc","fem"]
filas_brutas = len(raw)
dup = int(raw.duplicated(["cod","anio"]).sum()); nulos = int(raw.isna().sum().sum())
df = raw.drop_duplicates(["cod","anio"]).copy()
pob0 = int((df.pob <= 0).sum())
v = df[df.pob > 0]
chk = float((v.hi / v.pob * 1e5 - v.tasa).abs().max())
ANIOS = sorted(int(a) for a in df.anio.unique()); PARC = max(ANIOS); COMP = [a for a in ANIOS if a < PARC]
B, F = 2019, COMP[-1]
print(f"Filas brutas: {filas_brutas} | duplicados: {dup} | nulos: {nulos} | cantones: {df.cod.nunique()} | años {ANIOS[0]}-{ANIOS[-1]}")
print(f"Filas con población 0 (zonas sin población asignada, fuera de los rankings): {pob0}")
print(f"Máx. diferencia Tasa de la base vs HI/Pob×100.000: {chk:.2e}")
d = df[df.pob > 0].copy()          # cantones con población asignada
print("Cantones con población:", d.cod.nunique())

In [ ]:
# Regiones y grupo de referencia
REG = {}
for c in ["07","08","09","12","13","23","24"]: REG[c] = "Costa"
for c in ["01","02","03","04","05","06","10","11","17","18"]: REG[c] = "Sierra"
for c in ["14","15","16","19","21","22"]: REG[c] = "Amazonía"
REG["20"] = "Insular"
NOMBRES = {"DURAN":"Durán","QUITO":"Quito","LA MANA":"La Maná","MEJIA":"Mejía","PUJILI":"Pujilí","QUININDE":"Quinindé","SAMBORONDON":"Samborondón",
 "TULCAN":"Tulcán","ALAUSI":"Alausí","BIBLIAN":"Biblián","BOLIVAR":"Bolívar","BOLIVAR (MANABI)":"Bolívar (Manabí)","OLMEDO (MANABI)":"Olmedo (Manabí)",
 "CENTINELA DEL CONDOR":"Centinela del Cóndor","ECHEANDIA":"Echeandía","ESPINDOLA":"Espíndola","GIRON":"Girón","GONZANAMA":"Gonzanamá","JUNIN":"Junín",
 "LIMON INDANZA":"Limón Indanza","MARCABELI":"Marcabelí","NABON":"Nabón","PAJAN":"Paján","PUCARA":"Pucará","PUERTO LOPEZ":"Puerto López",
 "SAN CRISTOBAL":"San Cristóbal","SAN MIGUEL DE URCUQUI":"San Miguel de Urcuquí","SANTA LUCIA":"Santa Lucía","SANTIAGO DE PILLARO":"Santiago de Píllaro",
 "SAQUISILI":"Saquisilí","SIMON BOLIVAR":"Simón Bolívar","SUCUA":"Sucúa","CAMILO PONCE ENRIQUEZ":"Camilo Ponce Enríquez","SUCUMBIOS":"Sucumbíos",
 "MACARA":"Macará","GNRAL. ANTONIO ELIZALDE":"Gral. Antonio Elizalde","LOGROÑO":"Logroño"}
def nom(s):
    if s in NOMBRES: return NOMBRES[s]
    s = s.title()
    for a,b in [(" De "," de "),(" Del "," del "),(" Los "," los "),(" Las "," las "),(" La "," la ")]: s = s.replace(a,b)
    return s
d25 = d[d.anio == F].set_index("cod")
CORTE = 50000
REF = d25[d25.pob >= CORTE].index.tolist()
OTROS = [c for c in REF if c != "1701"]
print(f"Grupo de referencia: {len(REF)} cantones con >= {CORTE} hab. en {F} (Quito + {len(OTROS)})")
P = d.pivot(index="cod", columns="anio", values="tasa")
H = d.pivot(index="cod", columns="anio", values="hi")
POB = d.pivot(index="cod", columns="anio", values="pob")
FU = d.pivot(index="cod", columns="anio", values="fuego")

In [ ]:
# Posición de Quito por año (grupo fijo)
pos = {}
for a in ANIOS:
    q = P.loc["1701", a]; r = P.loc[REF, a]; o = P.loc[OTROS, a]
    ho, po = H.loc[OTROS, a].sum(), POB.loc[OTROS, a].sum()
    pos[a] = dict(quito=q, n_mayor=int((o > q).sum()), n_menor=int((o < q).sum()),
        mediana=float(r.median()), p25=float(r.quantile(.25)), p75=float(r.quantile(.75)),
        quito_pct_mediana=float(q / r.median() * 100), n_ge20=int((o >= 20).sum()), n_ge50=int((o >= 50).sum()),
        tasa_conjunta_otros=float(ho / po * 1e5))
print({a: (pos[a]["n_mayor"], round(pos[a]["quito_pct_mediana"])) for a in ANIOS})

In [ ]:
# Trayectorias 2019 → 2025 (los otros cantones frente a Quito)
q0, q1 = P.loc["1701", B], P.loc["1701", F]
def grupo(c):
    a, b = P.loc[c, B], P.loc[c, F]
    return ("pasaron_encima" if (a < q0 and b > q1) else "siguen_encima" if (a > q0 and b > q1)
            else "quedaron_debajo" if (a > q0 and b < q1) else "siguen_debajo" if (a < q0 and b < q1) else "otro")
G = {k: [] for k in ["pasaron_encima","siguen_encima","quedaron_debajo","siguen_debajo","otro"]}
for c in OTROS: G[grupo(c)].append(c)
def ordena(lst, col): return sorted(lst, key=lambda c: -P.loc[c, col])
grupos = {k: [dict(cod=c, name=nom(d25.loc[c,"canton"]), t0=float(P.loc[c,B]), t1=float(P.loc[c,F])) for c in ordena(lst, F)] for k, lst in G.items()}
# movimiento 2014 -> 2019
a0, a1 = P.loc["1701", 2014], q0
mov_14_19 = dict(superaron=int(sum(1 for c in OTROS if P.loc[c,2014] < a0 and P.loc[c,B] > a1)),
                 quedaron_debajo=int(sum(1 for c in OTROS if P.loc[c,2014] > a0 and P.loc[c,B] < a1)))
print({k: len(v) for k, v in G.items()}, mov_14_19)

In [ ]:
# Ritmo de cambio 2019 → 2025
veces = (P.loc[OTROS, F] / P.loc[OTROS, B].replace(0, np.nan))
qv = q1 / q0
menores_q = [c for c in OTROS if pd.notna(veces[c]) and veces[c] < qv]
ritmo = dict(n_aumento=int((P.loc[OTROS, F] > P.loc[OTROS, B]).sum()), n=len(OTROS), mediana_veces=float(veces.median()),
             n_ge5=int((veces >= 5).sum()), quito_veces=float(qv), n_menor_que_quito=len(menores_q),
             n_menor_parten_mas_alto=int(sum(1 for c in menores_q if P.loc[c,B] > q0)),
             tasa_conjunta_otros=[pos[B]["tasa_conjunta_otros"], pos[F]["tasa_conjunta_otros"]])

In [ ]:
# Cantones: base completa para tablas, mapa de calor y comparador
cant = []
for c in d25.index.tolist() + [x for x in P.index if x not in d25.index]:
    sub = d[d.cod == c]
    last = sub.iloc[-1]
    cant.append(dict(cod=c, name=nom(last.canton), provincia=last.provincia.title(), region=REG.get(last.prov, "s/d"),
        ref=bool(c in REF), pob=int(POB.loc[c, F]) if pd.notna(POB.loc[c, F]) else None,
        s={int(a): [float(P.loc[c,a]), int(H.loc[c,a]), int(FU.loc[c,a])] for a in ANIOS if pd.notna(P.loc[c,a])}))
ref_order = sorted(REF, key=lambda c: -P.loc[c, F])
cambio = [dict(cod=c, name=nom(d25.loc[c,"canton"]), region=REG.get(d25.loc[c,"prov"],"s/d"), pob=int(d25.loc[c,"pob"]),
               t0=float(P.loc[c,B]), t1=float(P.loc[c,F]), veces=(float(P.loc[c,F]/P.loc[c,B]) if P.loc[c,B] > 0 else None),
               dif=float(P.loc[c,F]-P.loc[c,B]), hi1=int(H.loc[c,F]), t26=float(P.loc[c,PARC]), hi26=int(H.loc[c,PARC])) for c in ref_order]

In [ ]:
# Ranking 2025, pequeños cantones y Pichincha
seguros = [dict(cod=c, name=nom(d25.loc[c,"canton"]), region=REG.get(d25.loc[c,"prov"],"s/d"), t1=float(P.loc[c,F]), hi1=int(H.loc[c,F]))
           for c in sorted([c for c in OTROS if P.loc[c,F] < q1], key=lambda c: P.loc[c,F])]
top = [dict(cod=c, name=nom(d25.loc[c,"canton"]), t1=float(P.loc[c,F]), hi1=int(H.loc[c,F]), region=REG.get(d25.loc[c,"prov"],"s/d")) for c in ref_order[:15]]
n50 = [c for c in OTROS if P.loc[c,F] >= 50]
reg50 = {r: int(sum(1 for c in n50 if REG.get(d25.loc[c,"prov"]) == r)) for r in ["Costa","Sierra","Amazonía","Insular"]}
peq = d25[d25.pob < CORTE]
menores = dict(n=len(peq), mayor_quito=int((peq.tasa > q1).sum()), sin_hi=int((peq.hi == 0).sum()), todos_sin_hi=int((d25.hi == 0).sum()), n_todos=len(d25))
pich = [dict(name=nom(r.canton), pob=int(r.pob), t1=float(r.tasa)) for _, r in d25[(d25.provincia == "PICHINCHA") & (d25.index != "1701")].sort_values("tasa", ascending=False).iterrows()]
ciu = d25[d25.pob >= 200000]
ciudades200 = dict(n=len(ciu), mayor=int((ciu.tasa > q1).sum()))
hi_ref = int(d25.loc[REF, "hi"].sum()); hi_tot = int(d25.hi.sum())

In [ ]:
# Composición y peso de Quito
comp = {}
nac_hi = df.groupby("anio").hi.sum(); nac_pob = df.groupby("anio").pob.sum()
for a in ANIOS:
    sq = d[(d.cod == "1701") & (d.anio == a)].iloc[0]; so = d[(d.cod.isin(OTROS)) & (d.anio == a)]
    comp[a] = dict(hi_q=int(sq.hi), fuego_q=float(sq.fuego/sq.hi*100), fuego_otros=float(so.fuego.sum()/so.hi.sum()*100),
        mujeres_q=float(sq.fem/(sq.masc+sq.fem)*100), mujeres_otros=float(so.fem.sum()/(so.masc.sum()+so.fem.sum())*100),
        peso_hi=float(sq.hi/nac_hi[a]*100), peso_pob=float(sq.pob/nac_pob[a]*100))

In [ ]:
# Salida APP_DATA
A = dict(meta=dict(filas_brutas=filas_brutas, duplicados=dup, nulos=nulos, cantones=int(df.cod.nunique()), cantones_con_pob=int(d.cod.nunique()),
        filas_pob0=pob0, anios=ANIOS, anio_parcial=int(PARC), anio_base=B, anio_fin=int(F), corte_ref=CORTE, n_ref=len(REF), n_otros=len(OTROS),
        delta_pob_2025=int(df[df.anio==F].pob.sum() - df[df.anio==F-1].pob.sum()), hi_ref_2025=hi_ref, hi_con_pob_2025=hi_tot,
        fuente="Base «Tasa_Homicidios_2020-2026_5.xlsx» (cantón × año)"),
    posicion=pos, grupos=grupos, mov_2014_2019=mov_14_19, ritmo=ritmo, cantones=cant, cambio=cambio, seguros=seguros, top=top,
    reg_ge50=reg50, n_ge50=len(n50), menores=menores, pichincha=pich, ciudades200=ciudades200, composicion=comp)
A = json.loads(json.dumps(A, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("APP_DATA listo")

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261014_homicidios_quito_vs_cantones_v3.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")